# Mission 3 · 증상 9클래스 multi-label (Colab GPU)로컬 CPU 베이스라인(TF-IDF + LogReg)은 macro-F1 **0.5848**. 이 노트북은 그 위에KLUE-RoBERTa / Kc-ELECTRA 파인튜닝을 올린다.**데이터 경로**: 로컬에서 `python -m src.preprocess.pack_for_colab --what m3` 로 만든`m3_text.json.gz` (14 MB) 를 Google Drive 의 `MyDrive/dcc/` 에 올려둘 것.전사 텍스트에는 개인정보가 포함되므로 **GitHub 에는 절대 올리지 않는다** — Drive 경유만 사용.

In [ ]:
import torch, subprocessprint(subprocess.run(['nvidia-smi','--query-gpu=name,memory.total','--format=csv'],                     capture_output=True, text=True).stdout)print('torch', torch.__version__, 'cuda', torch.cuda.is_available())

In [ ]:
from google.colab import drivedrive.mount('/content/drive')DATA = '/content/drive/MyDrive/dcc/m3_text.json.gz'CKPT = '/content/drive/MyDrive/dcc/ckpt'import os; os.makedirs(CKPT, exist_ok=True)

In [ ]:
!pip -q install "transformers>=4.44" "accelerate>=0.33" scikit-learn

In [ ]:
import gzip, json, numpy as npwith gzip.open(DATA, 'rt', encoding='utf-8') as f:    D = json.load(f)SYMPTOM_9 = ["고열","구토","두통","복통","어지러움","열상","오심","전신쇠약","호흡곤란"]tr, va = D['train'], D['val']Xtr = [r['text'] for r in tr];  Ytr = np.array([r['y'] for r in tr], dtype=np.float32)Xva = [r['text'] for r in va];  Yva = np.array([r['y'] for r in va], dtype=np.float32)# Training 안에서 dev 분리 — 임계값 튜닝 전용. Validation 은 학습/튜닝에 절대 사용 금지.rng = np.random.default_rng(0); perm = rng.permutation(len(tr)); n_dev = len(tr)//10dev_i, fit_i = perm[:n_dev], perm[n_dev:]print(f'fit={len(fit_i)}  dev={len(dev_i)}  val={len(Xva)}  labels={Ytr.shape}')print('클래스별 양성 비율:', dict(zip(SYMPTOM_9, (Ytr.mean(0)*100).round(1))))

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassificationMODEL = 'klue/roberta-base'      # 대안: 'beomi/KcELECTRA-base-v2022' (AI-Hub 공식 베이스라인 계열)MAXLEN = 512                      # 전체 대화 중앙값 397자 / p95 842자tok = AutoTokenizer.from_pretrained(MODEL)model = AutoModelForSequenceClassification.from_pretrained(    MODEL, num_labels=9, problem_type='multi_label_classification').cuda()

In [ ]:
import torchfrom torch.utils.data import Dataset, DataLoaderclass DS(Dataset):    def __init__(self, texts, y):        self.t, self.y = texts, y    def __len__(self):        return len(self.t)    def __getitem__(self, i):        return self.t[i], self.y[i]def collate(batch):    txt = [b[0] for b in batch]    y = torch.tensor(np.stack([b[1] for b in batch]))    enc = tok(txt, truncation=True, max_length=MAXLEN, padding=True, return_tensors='pt')    enc['labels'] = y    return encfit_dl = DataLoader(DS([Xtr[i] for i in fit_i], Ytr[fit_i]), batch_size=16,                    shuffle=True, collate_fn=collate, num_workers=2)dev_dl = DataLoader(DS([Xtr[i] for i in dev_i], Ytr[dev_i]), batch_size=32,                    collate_fn=collate, num_workers=2)val_dl = DataLoader(DS(Xva, Yva), batch_size=32, collate_fn=collate, num_workers=2)

In [ ]:
from torch.optim import AdamWfrom transformers import get_linear_schedule_with_warmupEPOCHS = 3opt = AdamW(model.parameters(), lr=2e-5, weight_decay=0.01)steps = len(fit_dl) * EPOCHSsched = get_linear_schedule_with_warmup(opt, int(0.06*steps), steps)scaler = torch.amp.GradScaler('cuda')@torch.no_grad()def predict(dl):    model.eval(); out = []    for b in dl:        b = {k: v.cuda() for k, v in b.items()}        with torch.amp.autocast('cuda'):            logits = model(**{k: v for k, v in b.items() if k != 'labels'}).logits        out.append(torch.sigmoid(logits.float()).cpu().numpy())    return np.concatenate(out)for ep in range(EPOCHS):    model.train(); tot = 0.0    for i, b in enumerate(fit_dl):        b = {k: v.cuda() for k, v in b.items()}        opt.zero_grad(set_to_none=True)        with torch.amp.autocast('cuda'):            loss = model(**b).loss        scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); sched.step()        tot += loss.item()        if (i+1) % 200 == 0:            print(f'ep{ep+1} step {i+1}/{len(fit_dl)} loss {tot/(i+1):.4f}', flush=True)    print(f'== epoch {ep+1} 평균 loss {tot/len(fit_dl):.4f}')

In [ ]:
from sklearn.metrics import f1_scoredef macro_f1(y_true, y_pred):    """출제 PDF 10쪽 정의: 클래스별 F1 → 단순 평균."""    return float(np.mean([f1_score(y_true[:,c], y_pred[:,c], zero_division=0)                          for c in range(y_true.shape[1])]))dev_s, val_s = predict(dev_dl), predict(val_dl)# 클래스별 임계값을 dev 에서 튜닝grid = np.arange(0.05, 0.95, 0.01)th = np.array([grid[np.argmax([f1_score(Ytr[dev_i][:,c], (dev_s[:,c]>=g).astype(int),                                        zero_division=0) for g in grid])]               for c in range(9)])pred05 = (val_s >= 0.5).astype(int)predth = (val_s >= th[None,:]).astype(int)print(f'Validation macro-F1  @0.5={macro_f1(Yva, pred05):.4f}   @tuned={macro_f1(Yva, predth):.4f}')print('로컬 CPU 베이스라인(TF-IDF+LogReg) = 0.5848')for c, s, t in zip(SYMPTOM_9, [f1_score(Yva[:,i], predth[:,i], zero_division=0) for i in range(9)], th):    print(f'  {c:6s} F1={s:.3f}  th={t:.2f}')

In [ ]:
import matplotlib.pyplot as pltimport matplotlib!apt-get -qq install fonts-nanum > /dev/nullmatplotlib.font_manager.fontManager.addfont('/usr/share/fonts/truetype/nanum/NanumGothic.ttf')matplotlib.rc('font', family='NanumGothic'); matplotlib.rc('axes', unicode_minus=False)fig, axes = plt.subplots(3, 3, figsize=(10.5, 10))for k, ax in enumerate(axes.ravel()):    t, p = Yva[:,k].astype(int), predth[:,k]    cm = np.array([[((t==0)&(p==0)).sum(), ((t==0)&(p==1)).sum()],                   [((t==1)&(p==0)).sum(), ((t==1)&(p==1)).sum()]], dtype=float)    pct = cm / np.maximum(cm.sum(1, keepdims=True), 1)    ax.imshow(pct, cmap='Blues', vmin=0, vmax=1)    ax.set_xticks([0,1], ['없음','있음']); ax.set_yticks([0,1], ['없음','있음'])    ax.set_title(f"{SYMPTOM_9[k]}  F1={f1_score(t, p, zero_division=0):.2f}", fontsize=10)    for i in range(2):        for j in range(2):            ax.text(j, i, f'{int(cm[i,j]):,}\n{pct[i,j]*100:.1f}%', ha='center', va='center',                    fontsize=9, color='white' if pct[i,j] > 0.55 else '#16202B')fig.suptitle(f'Mission 3 · KLUE-RoBERTa 혼동행렬 (macro-F1={macro_f1(Yva, predth):.4f})', fontsize=13)fig.tight_layout(); plt.show()

In [ ]:
torch.save({'state_dict': model.state_dict(), 'thresholds': th,            'model_name': MODEL, 'classes': SYMPTOM_9, 'maxlen': MAXLEN},           f'{CKPT}/m3.pt')print('저장 완료:', f'{CKPT}/m3.pt')